### Start 5G Network Simulator

In this DLI you will use an Agentic Generative AI solution to solve a bandwidth allocation problem. The lab will consist of two different parts. In the first part, the lab will show you how to setup an open source 5G Network Lab consisting of the following parts:
- 5G Core Lab simulation by Open Air Interface: https://openairinterface.org/oai-5g-core-network-project/
- FlexRIC that will be connected to the gNodeB and will be used to change the bandwidth allocation for each slice in the gNodeB
- RAN Lab composed by a gNodeB and two Use Equipment simulation components from Open Air OAI Softmode: https://github.com/simula/openairinterface5g/blob/dreibh/simulamet-testbed/doc/RUNMODEM.md
- Traffic generation over the Open Air network simulator using Iperf Tool: https://iperf.fr/

The Lab setup will start with the initialization of the 5G Core Network, then we will set up the gNodeB and the RIC connecting both via the E1 protocol. We will attach two UEs to the 5G network, each UE1 will have its own slice as seen in the diagram. Once UEs are functional, we will use the Iperf tool to generate traffic. First we will set up the Iperf server on the OAI External Network connected by the User Plane Function UPF.  Then we will use the Iperf Client to generate traffic against the External Network using the UEs connection. 

![5G Lab](./DLI_Picture.jpg)

In this Jupyter Notebook we will set the lab for the experiment. In a separate Jupyter Notebook we will build the Agentic Workflow for the experiment.

### OAI 5G Network 
To set up the 5G core Network funcitons we will use the docker compose comamand. Firt we will setup the standard network funciton for the core and then we will set up an additonal slice that will have its own SMF and UPF

In [11]:
!docker compose -f docker-compose-oai-cn-slice1.yaml up -d
import time
time.sleep(20)
!docker compose -f docker-compose-oai-cn-slice2.yaml up -d


WARN[0000] /home/nvidia/llm-slicing-5g-lab/docker-compose-oai-cn-slice1.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 


[+] Running 10/0
 ✔ Network demo-oai-public-net  Created                                    0.0s 
 ✔ Container mysql              Created                                    0.0s 
 ✔ Container oai-ext-dn         Created                                    0.0s 
 ✔ Container oai-nrf            Created                                    0.0s 
 ✔ Container oai-nssf           Created                                    0.0s 
 ✔ Container oai-pcf            Created                                    0.0s 
 ✔ Container oai-udr            Created                                    0.0s 
 ✔ Container oai-udm            Created                                    0.0s 
 ✔ Container oai-ausf           Created                                    0.0s 
 ✔ Container oai-amf            Created                                    0.0s 
 ⠋ Container oai-smf-slice1     Creating                                   0.0s 
[+] Running 8/11
 ✔ Network demo-oai-public-net  Created                                    

### Start RIC

Then we will start the FlexRIC to be able to modify parameters in the gNodeB

In [12]:
import subprocess

cmd = "./flexric/build/examples/ric/nearRT-RIC"
logfile = "logs/RIC.log"

# Open log file for writing and start the process
with open(logfile, "a") as log:
    process = subprocess.Popen(
        ["bash", "-c", f"stdbuf -oL {cmd}"],  # stdbuf ensures real-time logging
        stdout=log,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )

print(f"Process started with PID {process.pid}, logging to {logfile}.")

Process started with PID 3330889, logging to logs/RIC.log.


### Start the gNodeB
Then we will run the gNodeB using the OAI softmoden software

In [13]:
import subprocess

cmd = "sudo LD_LIBRARY_PATH=. ./openairinterface5g/cmake_targets/ran_build/build/nr-softmodem -O ran-conf/gnb.conf --sa --rfsim -E --gNBs.[0].min_rxtxtime 6"
logfile = "logs/gNodeB.log"

# Open log file for writing and start the process
with open(logfile, "a") as log:
    process = subprocess.Popen(
        ["bash", "-c", f"stdbuf -oL {cmd}"],  # stdbuf ensures real-time logging
        stdout=log,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )

print(f"Process started with PID {process.pid}, logging to {logfile}.")


Process started with PID 3331191, logging to logs/gNodeB.log.


### Initialize Bandwidth 50 50 
The gNodeB will have initially allocated 50% of its bandwidth to each of the slices. 

In [14]:
!./change_rc_slice.sh 50 50

50+50
[UTIL]: Setting the config -c file to /usr/local/etc/flexric/flexric.conf
[UTIL]: Setting path -p for the shared libraries to /usr/local/lib/flexric/
[xAapp]: Initializing ... 
[xApp]: nearRT-RIC IP Address = 127.0.0.1, PORT = 36422
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/libslice_sm.so 
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/libpdcp_sm.so 
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/libkpm_sm.so 
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/libtc_sm.so 
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/libgtp_sm.so 
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/librlc_sm.so 
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/librc_sm.so 
[E2 AGENT]: Opening plugin from path = /usr/local/lib/flexric/libmac_sm.so 
[NEAR-RIC]: Loading SM ID = 145 with def = SLICE_STATS_V0 
[NEAR-RIC]: Loading SM ID = 144 with def = PDCP_STATS_V0 
[NEAR-RIC]: Loading SM ID = 2 with 

Connected E2 nodes = 1
Setting PRB Ratio to 50:50
[xApp]: CONTROL-REQUEST tx 
[xApp]: CONTROL ACK rx
[xApp]: Successfully received CONTROL-ACK 
[xApp]: Control Loop Latency: 707 us
[xApp]: Sucessfully stopped 
Test xApp run SUCCESSFULLY


### Start the UE1 
After we will create a User Equipment Simulator and attach it to the gNodeB

In [15]:
import subprocess

# Define command sequence
cmd = """
sudo ./multi_ue.sh -c1 -e & 
sleep 5  # Give time for namespace setup
sudo ip netns exec ue1 bash -c '
    LD_LIBRARY_PATH=. ./openairinterface5g/cmake_targets/ran_build/build/nr-uesoftmodem \
    --rfsimulator.serveraddr 10.201.1.100 -r 106 --numerology 1 --band 78 -C 3619200000 \
    --rfsim --sa -O ran-conf/ue_1.conf -E
'
"""

logfile = "logs/UE1.log"

# Open log file for writing and start the process
with open(logfile, "a") as log:
    process = subprocess.Popen(
        ["bash", "-c", f"stdbuf -oL {cmd}"],  # stdbuf ensures real-time logging
        stdout=log,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )

print(f"Process started with PID {process.pid}, logging to {logfile}.")


Process started with PID 3332721, logging to logs/UE1.log.


### Start UE2 

We will do the same with second User Equipment that will be attach to the gNodeB in the second slice

In [16]:
import subprocess

# Define command sequence
cmd = """
sudo ./multi_ue.sh -c3 -e & 
sleep 5  # Give time for namespace setup
sudo ip netns exec ue3 bash -c '
    LD_LIBRARY_PATH=. ./openairinterface5g/cmake_targets/ran_build/build/nr-uesoftmodem \
    --rfsimulator.serveraddr 10.203.1.100 -r 106 --numerology 1 --band 78 -C 3619200000 \
    --rfsim --sa -O ran-conf/ue_2.conf -E
'
"""

logfile = "logs/UE2.log"

# Open log file for writing and start the process
with open(logfile, "a") as log:
    process = subprocess.Popen(
        ["bash", "-c", f"stdbuf -oL {cmd}"],  # stdbuf ensures real-time logging
        stdout=log,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )

print(f"Process started with PID {process.pid}, logging to {logfile}.")


Process started with PID 3333327, logging to logs/UE2.log.


### Start the Iperf Tool Server for UE1
Once the 5G Network Simulation is running we will start the simulation of traffic by using with tool Iperf. First we will created an Iperf Server that will be on the External Network connected via the User Plane Function. 

In [17]:
import subprocess

cmd = "docker exec -t oai-ext-dn iperf3 -s -p 5201"
logfile = "logs/docker_iperfserver1.log"

# Open log file for writing and start the process
with open(logfile, "a") as log:
    process = subprocess.Popen(
        ["bash", "-c", f"stdbuf -oL {cmd}"],  # stdbuf ensures real-time logging
        stdout=log,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )

print(f"Process started with PID {process.pid}, logging to {logfile}.")

Process started with PID 3333695, logging to logs/docker_iperfserver1.log.


### Start the Iperf Tool Server for UE2
Then we will create another IperfServer for the second User Equipment UE2

In [18]:
import subprocess

cmd = "docker exec -t oai-ext-dn iperf3 -s -p 5202"
logfile = "logs/docker_iperfserver2.log"

# Open log file for writing and start the process
with open(logfile, "a") as log:
    process = subprocess.Popen(
        ["bash", "-c", f"stdbuf -oL {cmd}"],  # stdbuf ensures real-time logging
        stdout=log,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )

print(f"Process started with PID {process.pid}, logging to {logfile}.")

Process started with PID 3334162, logging to logs/docker_iperfserver2.log.


## Start Traffic Generator
Once every element in The Network is up and running and the Iperf server is listening in the external network. We will run two iperf clients that will be generating traffic in both UEs. These scripts will generate udp traffic from the iperf server towards the UE and will alternate speeds 30M and 120M for 100 seconds.

In [19]:
import subprocess
import os

# Ensure the logs directory exists
log_dir = "logs"

# Define commands and log files
cmds = [
    ("./traffic_generator_ue1.sh", "logs/traffic_generator_ue1.log"),
    ("./traffic_generator_ue2.sh", "logs/traffic_generator_ue2.log")
]

processes = []

for cmd, logfile in cmds:
    with open(logfile, "a") as log:
        process = subprocess.Popen(
            ["bash", "-c", f"stdbuf -oL {cmd}"],  # Ensure real-time logging
            stdout=log,
            stderr=subprocess.STDOUT,
            universal_newlines=True
        )
    processes.append((cmd, process.pid))
    print(f"Started {cmd} with PID {process.pid}, logging to {logfile}")

# Save PIDs for reference
pid_file = "logs/processes.csv"
with open(pid_file, "a") as pid_log:
    for cmd, pid in processes:
        pid_log.write(f"{pid},{cmd}\n")

print(f"All processes launched. PIDs saved in {pid_file}.")


Started ./traffic_generator_ue1.sh with PID 3334804, logging to logs/traffic_generator_ue1.log
Started ./traffic_generator_ue2.sh with PID 3334805, logging to logs/traffic_generator_ue2.log
All processes launched. PIDs saved in logs/processes.csv.


This will be the final step setting up the 5G Lab for the Agentic Workflow.

![stop](./Stop2.jpg) 

## Shutting Down the Lab
This section is just in case you need to restart the lab. Do not execute if your lab is running properly. 


1. Shutdown traffic Generator
2. Shutdown iperf Server
3. Shutdown UEs
4. Shutdown eNodeB
5. Shutdown RIC
6. Shutdown 5G Core
7. Remove Name Spaces

### Shutdown traffic Generator

In [11]:
#Stop Traffic Generator EU1

import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "traffic_generator_ue1.sh"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'traffic_generator_ue1.sh' processes have been terminated.")
        else:
            print("No 'traffic_generator_ue1.sh' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()

No 'traffic_generator_ue1.sh' processes found.


In [12]:
#Stop Traffic Generator EU2

import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "traffic_generator_ue2.sh"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'traffic_generator_ue2.sh' processes have been terminated.")
        else:
            print("No 'traffic_generator_ue2.sh' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()

No 'traffic_generator_ue2.sh' processes found.


###  Shutdown iperf Server


In [13]:
#Stop Iperf Server 1
import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "iperf3 -s -p 5201"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'iperf3 -s -p 5201' processes have been terminated.")
        else:
            print("No 'iperf3 -s -p 5201' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()

Killing process with PID: 2392224
Killing process with PID: 2392285
All 'iperf3 -s -p 5201' processes have been terminated.


In [14]:
import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "iperf3 -s -p 5202"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'iperf3 -s -p 5202' processes have been terminated.")
        else:
            print("No 'iperf3 -s -p 5202' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()

Killing process with PID: 2392225
Killing process with PID: 2392278
All 'iperf3 -s -p 5202' processes have been terminated.


###  Shutdown UEs

In [15]:
#Stop UE1
import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "ue_1"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'ue_1' processes have been terminated.")
        else:
            print("No 'ue_1' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()

Killing process with PID: 2376736
Killing process with PID: 2377623
Killing process with PID: 2377628
Killing process with PID: 2377630
Killing process with PID: 2377633
Killing process with PID: 2377638
All 'ue_1' processes have been terminated.


In [16]:
#Stop UE2 

import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "ue_2"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'ue_2' processes have been terminated.")
        else:
            print("No 'ue_2' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()

Killing process with PID: 2376740
Killing process with PID: 2377624
Killing process with PID: 2377629
Killing process with PID: 2377632
Killing process with PID: 2377634
Killing process with PID: 2377642
All 'ue_2' processes have been terminated.


###  Shutdown eNodeB

In [17]:
#Kill the gNodeB = Work on this
import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "gnb.conf"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'gnb.conf' processes have been terminated.")
        else:
            print("No 'gnb.conf' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()

Killing process with PID: 2373280
Killing process with PID: 2373281
Killing process with PID: 2373282
All 'gnb.conf' processes have been terminated.


### Shutdown RIC

In [18]:
#Kill the RIC process
import subprocess

def kill_openair_processes():
    try:
        # Get list of all running processes containing "openair" in the command
        result = subprocess.run(["pgrep", "-f", "nearRT-RIC"], capture_output=True, text=True)

        if result.stdout.strip():
            pids = result.stdout.strip().split("\n")
            for pid in pids:
                print(f"Killing process with PID: {pid}")
                subprocess.run(["sudo", "kill", "-9", pid], check=True)
            print("All 'nearRT-RIC' processes have been terminated.")
        else:
            print("No 'nearRT-RIC' processes found.")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    kill_openair_processes()


Killing process with PID: 2372635
All 'nearRT-RIC' processes have been terminated.


### Stop 5G Network 

In [19]:
!docker compose -f docker-compose-oai-cn-slice1.yaml down
!docker compose -f docker-compose-oai-cn-slice2.yaml down

WARN[0000] /home/nvidia/x-fx-84-v1/task1/notebooks/llm-slicing-5g-lab/docker-compose-oai-cn-slice1.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 


[+] Running 0/1
 ⠋ Container oai-ausf  Stopping                                            0.1s 
[+] Running 1/2
 ✔ Container oai-ausf  Removed                                             0.2s 
 ⠋ Container oai-udm   Stopping                                            0.0s 
[+] Running 1/2
 ✔ Container oai-ausf  Removed                                             0.2s 
 ⠙ Container oai-udm   Stopping                                            0.1s 
[+] Running 1/2
 ✔ Container oai-ausf  Removed                                             0.2s 
 ⠹ Container oai-udm   Stopping                                            0.2s 
[+] Running 1/2
 ✔ Container oai-ausf  Removed                                             0.2s 
 ⠸ Container oai-udm   Stopping                                            0.3s 
[+] Running 1/2
 ✔ Container oai-ausf  Removed                                             0.2s 
 ⠼ Container oai-udm   Stopping                                            0.4s 
[+] Running 1

## Remove name spaces

In [20]:
!sudo ./multi_ue.sh -d1 
!sudo ./multi_ue.sh -d3

deleting namespace for UE ID 1 name ue1


deleting namespace for UE ID 3 name ue3


### Reset Logs and Database

In [10]:
import os

# List of log files to reset (empty)
log_files = [
    "logs/docker_iperfserver2.log",
    "logs/traffic_generator_ue1.log",
    "logs/traffic_generator_ue2.log",
    "logs/UE1_iperfc.log",
    "logs/docker_iperfserver1.log",
    "logs/UE1.log",
    "logs/UE2.log",
    "logs/gNodeB.log",
    "logs/RIC.log",
    "logs/UE2_iperfc.log",
]

# Function to empty log files
def reset_logs(log_files):
    for file in log_files:
        try:
            with open(file, "w") as f:
                f.write("")  # Empty the file
            print(f"✅ Successfully emptied: {file}")
        except Exception as e:
            print(f"❌ Error emptying {file}: {e}")

# Run the function
if __name__ == "__main__":
    reset_logs(log_files)


✅ Successfully emptied: logs/docker_iperfserver2.log
✅ Successfully emptied: logs/traffic_generator_ue1.log
✅ Successfully emptied: logs/traffic_generator_ue2.log
✅ Successfully emptied: logs/UE1_iperfc.log
✅ Successfully emptied: logs/docker_iperfserver1.log
✅ Successfully emptied: logs/UE1.log
✅ Successfully emptied: logs/UE2.log
✅ Successfully emptied: logs/gNodeB.log
✅ Successfully emptied: logs/RIC.log
✅ Successfully emptied: logs/UE2_iperfc.log


## Flush the database

In [ ]:
import sqlite3

db_path = "performance.db"

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

try:
    # Delete all rows from log_entries table
    cursor.execute("DELETE FROM log_entries;")
    conn.commit()
    print("All rows removed from log_entries table.")
except sqlite3.Error as e:
    print("SQLite error:", e)
finally:
    # Close the connection
    conn.close()


All rows removed from log_entries table.


In [ ]:
import sqlite3

db_path = "performance.db"

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

try:
    # Delete all rows from log_entries table
    cursor.execute("DELETE FROM log_entries;")
    conn.commit()
    print("All rows removed from log_entries table.")
except sqlite3.Error as e:
    print("SQLite error:", e)
finally:
    # Close the connection
    conn.close()


All rows removed from log_entries table.


In [ ]:
import sqlite3

DB_FILE = "performance.db"

def show_table_info():
    """Displays the table structure and first few rows from the log_entries table."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    # Fetch column names
    cursor.execute("PRAGMA table_info(log_entries);")
    columns = cursor.fetchall()

    print("\n📌 Table Columns:")
    for col in columns:
        print(f"- {col[1]} ({col[2]})")

    # Fetch and display first 10 rows
    print("\n📌 First 10 Rows in log_entries:")
    cursor.execute("SELECT * FROM log_entries ORDER BY id DESC LIMIT 10;")
    rows = cursor.fetchall()

    for row in rows:
        print(row)

    conn.close()

if __name__ == "__main__":
    show_table_info()



📌 Table Columns:
- id (INTEGER)
- ue (TEXT)
- timestamp (DATETIME)
- stream (INTEGER)
- interval_start (REAL)
- interval_end (REAL)
- duration (REAL)
- data_transferred (REAL)
- bitrate (REAL)
- jitter (REAL)
- lost_packets (INTEGER)
- total_packets (INTEGER)
- loss_percentage (REAL)

📌 First 10 Rows in log_entries:


In [39]:
!pip install influxdb

from influxdb import InfluxDBClient
import time

# Configuration
host = '127.0.0.1'  # This is where your InfluxDB is running on your local machine
port = 8086         # Default InfluxDB port
username = 'admin'  # Default username (if authentication is enabled)
password = 'admin'  # Default password (if authentication is enabled)
database = 'test_db'  # Name of the database to create or use

# Connect to InfluxDB
client = InfluxDBClient(host=host, port=port, username=username, password=password)

# Create a database (if it doesn't exist)
client.create_database(database)

# Switch to the test database
client.switch_database(database)

# Write data into the InfluxDB
json_body = [
    {
        "measurement": "test_table",  # This is the equivalent of a table
        "tags": {
            "host": "localhost"  # You can use tags to categorize the data (optional)
        },
        "fields": {
            "column_1": 25.6,  # First column
            "column_2": 45.2   # Second column
        }
    }
]

# Write the data point to the database
client.write_points(json_body)

# Wait for a moment to ensure data is written
time.sleep(1)

# Query the data to verify it's been written
result = client.query('SELECT * FROM "test_table"')

# Print the results
print("Query result:", result)

# Close the connection
client.close()

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Query result: ResultSet({'('test_table', None)': [{'time': '2025-04-09T20:13:57.325758Z', 'column_1': 25.6, 'column_2': 45.2, 'host': 'localhost'}]})
